# Train the fastener detector

This notebook trains a YOLO object detector on the pictures you labelled in the **Label** tab of the Tool Scale site, then exports `fasteners.onnx` and `classes.json` for the **Detect** tab.

**Before you start:** in Colab choose *Runtime > Change runtime type > T4 GPU* (free).

Steps: run the cells top to bottom. When asked, upload `fastener-dataset.zip` (the file the Label tab exported).

*Licence note:* this uses the open-source Ultralytics YOLO library (AGPL-3.0). That is fine for internal use. If you ever sell a product that includes it, read the licence or consider an Apache-2.0 detector such as YOLOX.

In [3]:
!pip -q install ultralytics onnx onnxruntime
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (training will be slow; choose a GPU runtime)")

GPU: Tesla T4


## 1. Upload the dataset

In [ ]:
from google.colab import files
import zipfile, os, json, glob, shutil
up = files.upload()                       # choose fastener-dataset.zip
name = next(iter(up))
shutil.rmtree('/content/dataset', ignore_errors=True)
zipfile.ZipFile(name).extractall('/content/dataset')
meta = json.load(open('/content/dataset/classes.json'))
names = [n['name'] for n in meta['names']]
print(len(names), 'classes:', names)
print('train pictures:', len(glob.glob('/content/dataset/images/train/*.jpg')), ' val pictures:', len(glob.glob('/content/dataset/images/val/*.jpg')))

# count boxes per class so you can see if any type is under-labelled
counts = {i: 0 for i in range(len(names))}
for f in glob.glob('/content/dataset/labels/*/*.txt'):
    for line in open(f).read().splitlines():
        if line.strip(): counts[int(line.split()[0])] += 1
for i, n in enumerate(names): print(f'{counts[i]:5d} boxes  {n}', '   <-- label more (30+ needed)' if counts[i] < 30 else '')

# write an absolute-path data file for training
with open('/content/dataset/data_abs.yaml', 'w') as f:
    f.write('path: /content/dataset\ntrain: images/train\nval: images/val\nnc: %d\nnames: %s\n' % (len(names), json.dumps(names)))

## 2. Train
Starts from a small model already trained on millions of pictures (`yolov8n`), so a few hundred labelled boxes are enough. Rotation and flip augmentation are on because fasteners can lie any way round.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='/content/dataset/data_abs.yaml', epochs=80, imgsz=640, batch=16, patience=25,
            project='/content/runs', name='fasteners', exist_ok=True,
            degrees=180, flipud=0.5, fliplr=0.5, hsv_v=0.4, hsv_s=0.5, scale=0.4, mosaic=1.0)

## 3. How good is it?
mAP50 close to 1.0 is excellent. The confusion matrix shows which types get mixed up. If two types are confused, label more pictures of both.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
save_dir = Path(model.trainer.save_dir)
best = YOLO(save_dir / 'weights' / 'best.pt')
m = best.val(data='/content/dataset/data_abs.yaml')
print('mAP50:', round(float(m.box.map50), 3), ' mAP50-95:', round(float(m.box.map), 3))
display(Image(str(save_dir / 'confusion_matrix_normalized.png')))

Look at a few predictions on the validation pictures:

In [ ]:
import random
vals = glob.glob('/content/dataset/images/val/*.jpg')
for p in random.sample(vals, min(4, len(vals))):
    r = best.predict(p, conf=0.35, imgsz=640, verbose=False)[0]
    r.save(filename='/content/pred.jpg'); display(Image('/content/pred.jpg'))

## 4. Export for the website

In [ ]:
out = best.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
os.makedirs('/content/out', exist_ok=True)
shutil.copy(out, '/content/out/fasteners.onnx')
shutil.copy('/content/dataset/classes.json', '/content/out/classes.json')
print(os.listdir('/content/out'), round(os.path.getsize('/content/out/fasteners.onnx')/1e6, 1), 'MB')
files.download('/content/out/fasteners.onnx')
files.download('/content/out/classes.json')

## 5. Put it on the site

1. Open your GitHub repository (`toolmanagment`) in the browser.
2. Open the `models` folder, choose *Add file > Upload files*, and upload **both** `fasteners.onnx` and `classes.json`. Commit to `main`.
3. Wait a minute or two, then open the **Detect** tab. It loads the model and counts the fasteners in view.

To improve it later: label more pictures (especially ones it got wrong), export again, retrain, and upload the new files.